In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split,cross_val_score
from sklearn.preprocessing import StandardScaler,MinMaxScaler
from sklearn.model_selection import GridSearchCV,StratifiedKFold
import pickle
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score
from sklearn.compose import ColumnTransformer

In [ ]:
df = pd.read_csv("data.csv")

In [ ]:
df.shape

In [ ]:
df.sample(4)

In [ ]:
df.columns

## Pre processing:

In [ ]:
df.isnull().sum()

In [ ]:
df.duplicated().sum()

In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
df.drop(columns=['id','diagnosis'],inplace=True)

In [ ]:
df.sample()

In [ ]:
Trans = ColumnTransformer(transformers=[  
    ('trf1', MinMaxScaler(), list(range(0, 30)))  
])


In [ ]:
df = Trans.fit_transform(df)

In [ ]:
[x for x in df.columns if type(x)=='str']

In [ ]:
df.shape

In [ ]:
df

## KMeans Clustering:

In [ ]:
from sklearn.cluster import KMeans
kmeans = KMeans(n_clusters=2, random_state=42)
kmeans.fit(df)
pred_labels = kmeans.labels_

# Hyperparameter tuning using Elbow Method
wcss = []
max_k = 10
for k in range(1, max_k + 1):
    kmeans = KMeans(n_clusters=k, random_state=42)
    kmeans.fit(df)
    wcss.append(kmeans.inertia_)

# Plot the elbow curve
plt.plot(range(1, max_k + 1), wcss, marker='o')
plt.title('Elbow Method for Optimal k (KMeans)')
plt.xlabel('Number of Clusters (k)')
plt.ylabel('Within-Cluster Sum of Squares (WCSS)')
plt.xticks(range(1, max_k + 1))
plt.grid(True)
plt.show()

# Calculate accuracy (not applicable for clustering, just for demonstration)
# Accuracy calculation is usually not meaningful for clustering
accuracy_before_tuning = kmeans.score(df)
print("Accuracy before hyperparameter tuning (WCSS):", accuracy_before_tuning)

In [ ]:
kmeans = KMeans(n_clusters=15, random_state=42)
kmeans.fit(df)
wcss_before_tuning = kmeans.inertia_

print("WCSS before hyperparameter tuning:", wcss_before_tuning)

In [ ]:
from sklearn.cluster import AgglomerativeClustering
from sklearn.metrics import silhouette_score
from sklearn.model_selection import GridSearchCV

# Define a custom scoring function for silhouette score
def silhouette_scorer(estimator, X):
    labels = estimator.fit_predict(X)
    return silhouette_score(X, labels)

# Assuming df is your DataFrame containing feature columns

# Training Agglomerative Clustering without hyperparameter tuning
agg = AgglomerativeClustering(n_clusters=2)
agg.fit(df)

# Calculate silhouette score before tuning
silhouette_score_before_tuning = silhouette_score(df, agg.labels_)
print("Silhouette score before hyperparameter tuning:", silhouette_score_before_tuning)

# Hyperparameter tuning using GridSearchCV with custom scoring function
param_grid = {
    'n_clusters': [2, 3, 4, 5],
    'linkage': ['ward', 'complete', 'average', 'single']
}

grid_search = GridSearchCV(AgglomerativeClustering(), param_grid, scoring=silhouette_scorer, cv=5)
grid_search.fit(df)

# Print the best parameters found
print("Best parameters:", grid_search.best_params_)

# Training Agglomerative Clustering with best hyperparameters
best_agg = grid_search.best_estimator_
best_agg.fit(df)

# Calculate silhouette score after tuning
silhouette_score_after_tuning = silhouette_score(df, best_agg.labels_)
print("Silhouette score after hyperparameter tuning:", silhouette_score_after_tuning)


In [ ]:
from sklearn.cluster import DBSCAN
from sklearn.metrics import silhouette_score

# Assuming df is your DataFrame containing feature columns

# Training DBSCAN without hyperparameter tuning
dbscan = DBSCAN(eps=0.5, min_samples=5)
dbscan.fit(df)

# Calculate silhouette score before tuning
silhouette_score_before_tuning = silhouette_score(df, dbscan.labels_)
print("Silhouette score before hyperparameter tuning:", silhouette_score_before_tuning)

# Hyperparameter tuning (not applicable for DBSCAN)

# Training DBSCAN without hyperparameter tuning (no changes)
dbscan_optimized = dbscan

# Calculate silhouette score after tuning (no changes)
silhouette_score_after_tuning = silhouette_score(df, dbscan_optimized.labels_)
print("Silhouette score after hyperparameter tuning (no changes):", silhouette_score_after_tuning)



In [ ]:
import matplotlib.pyplot as plt
from sklearn.cluster import AgglomerativeClustering
from sklearn.metrics import silhouette_score
from sklearn.model_selection import GridSearchCV

# Assuming df is your DataFrame containing feature columns

# Define a custom scoring function for silhouette score
def silhouette_scorer(estimator, X):
    labels = estimator.fit_predict(X)
    return silhouette_score(X, labels)

# Hyperparameter tuning using GridSearchCV with custom scoring function
param_grid = {
    'n_clusters': [2, 3, 4, 5],
    'linkage': ['ward', 'complete', 'average', 'single']
}

grid_search = GridSearchCV(AgglomerativeClustering(), param_grid, scoring=silhouette_scorer, cv=5)
grid_search.fit(df)

# Print the best parameters found
print("Best parameters:", grid_search.best_params_)

# Training Agglomerative Clustering with best hyperparameters
best_agg = grid_search.best_estimator_
best_agg.fit(df)

# Visualize clusters
plt.scatter(df[:, 0], df[:, 1], c=best_agg.labels_, cmap='viridis')
plt.title('Agglomerative Clustering with Best Hyperparameters')
plt.colorbar(label='Cluster Label')
plt.show()
